# Phase 1: Pendulum-v1 Experiments

**Notebook:** `10_pendulum_experiments.ipynb`  
**Phase:** 1 - Simple Control  
**Author:** Saurabh Jalendra  

## Objectives
1. Run all 5 quantum-inspired approaches on Pendulum-v1
2. Collect multi-seed results (5 seeds each)
3. Evaluate test set performance and long-horizon prediction
4. Save results to experiments/results/phase1/pendulum/

In [1]:
"""
Cell: Imports and Configuration
Purpose: Set up environment, imports, and experiment configuration
"""
import sys
import os
import json
import time
import traceback
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, NamedTuple
from dataclasses import dataclass, field
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Import quantum-inspired components
from quantum_inspired.tunneling_optimizer import QuantumTunnelingOptimizer
from quantum_inspired.superposition_buffer import SuperpositionReplayBuffer
from quantum_inspired.entanglement_layer import EntanglementLayer
from quantum_inspired.interference_ensemble import InterferenceEnsemble

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configuration
OBS_DIM = 3
ACTION_DIM = 1
STOCH_DIM = 64
DETER_DIM = 512
HIDDEN_DIM = 512
STATE_DIM = DETER_DIM + STOCH_DIM

NUM_STEPS = 10000
BATCH_SIZE = 32
SEQ_LEN = 20
LEARNING_RATE = 3e-4
KL_WEIGHT = 1.0
GRAD_CLIP = 100.0
NUM_EPISODES = 100
EXPERIMENT_SEEDS = [42, 123, 456, 789, 1024]

APPROACHES = ["baseline", "quantum_tunneling", "superposition", "entanglement", "interference_ensemble"]

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results" / "phase1" / "pendulum"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration: obs_dim={OBS_DIM}, action_dim={ACTION_DIM}")
print(f"Architecture: stoch={STOCH_DIM}, deter={DETER_DIM}, hidden={HIDDEN_DIM}")
print(f"Training: steps={NUM_STEPS}, batch={BATCH_SIZE}, seq_len={SEQ_LEN}, lr={LEARNING_RATE}")
print(f"Seeds: {EXPERIMENT_SEEDS}")
print(f"Results directory: {RESULTS_DIR}")

Device: cuda
GPU: NVIDIA GeForce RTX 5090
Configuration: obs_dim=3, action_dim=1
Architecture: stoch=64, deter=512, hidden=512
Training: steps=10000, batch=32, seq_len=20, lr=0.0003
Seeds: [42, 123, 456, 789, 1024]
Results directory: d:\Git Repos\Quantum-Enhanced-Simulation-Learning-for-Reinforcement-Learning\experiments\results\phase1\pendulum


---
## Data Collection

Collect Pendulum-v1 episodes using a random policy.  
Pendulum-v1: obs_dim=3 (cos, sin, angular velocity), action_dim=1 (torque in [-2, 2]), max_steps=200.

In [2]:
"""
Cell: Data Collection and Replay Buffer
Purpose: Collect Pendulum episodes and create replay buffer
"""

def set_seed(seed):
    """Set all random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def collect_pendulum_episodes(num_episodes, seed):
    """Collect episodes from Pendulum-v1 using random policy.

    Pendulum-v1 has continuous actions in [-2, 2] and observations
    of shape (3,): [cos(theta), sin(theta), theta_dot].
    Episodes have a fixed length of 200 steps (no early termination).

    Parameters
    ----------
    num_episodes : int
        Number of episodes to collect
    seed : int
        Random seed for environment and action sampling

    Returns
    -------
    List[Dict]
        List of episode dictionaries with keys:
        'obs', 'actions', 'rewards', 'dones'
    """
    env = gym.make("Pendulum-v1")
    episodes = []
    rng = np.random.RandomState(seed)

    for i in range(num_episodes):
        obs, info = env.reset(seed=seed + i)
        obs_list, act_list, rew_list, done_list = [], [], [], []

        terminated = False
        truncated = False
        while not terminated and not truncated:
            obs_list.append(np.array(obs, dtype=np.float32))
            # Random continuous action in [-2, 2]
            action = rng.uniform(-2.0, 2.0, size=(1,)).astype(np.float32)
            obs, reward, terminated, truncated, info = env.step(action)
            act_list.append(action)
            rew_list.append(float(reward))
            done_list.append(float(terminated or truncated))

        if len(obs_list) > 0:
            episodes.append({
                "obs": np.array(obs_list, dtype=np.float32),
                "actions": np.array(act_list, dtype=np.float32),
                "rewards": np.array(rew_list, dtype=np.float32),
                "dones": np.array(done_list, dtype=np.float32),
            })

    env.close()
    return episodes


class ReplayBuffer:
    """Simple episode replay buffer for standard and non-superposition approaches."""

    def __init__(self, capacity=10000):
        self.episodes = []
        self.capacity = capacity

    def add(self, episode):
        if len(self.episodes) >= self.capacity:
            self.episodes.pop(0)
        self.episodes.append(episode)

    def sample(self, batch_size, seq_len):
        """Sample batch of sequences from stored episodes.

        Returns
        -------
        Tuple of numpy arrays: (obs, actions, rewards, dones)
            obs:     (batch_size, seq_len, obs_dim)
            actions: (batch_size, seq_len, action_dim)
            rewards: (batch_size, seq_len)
            dones:   (batch_size, seq_len)
        """
        obs_b, act_b, rew_b, done_b = [], [], [], []
        for _ in range(batch_size):
            ep = self.episodes[np.random.randint(len(self.episodes))]
            L = len(ep["obs"])
            if L <= seq_len:
                pad = seq_len - L
                obs = np.pad(ep["obs"], ((0, pad), (0, 0)), mode="edge")
                act = np.pad(ep["actions"], ((0, pad), (0, 0)), mode="edge")
                rew = np.pad(ep["rewards"], (0, pad), mode="edge")
                done = np.pad(ep["dones"], (0, pad), mode="edge")
            else:
                s = np.random.randint(0, L - seq_len)
                obs = ep["obs"][s : s + seq_len]
                act = ep["actions"][s : s + seq_len]
                rew = ep["rewards"][s : s + seq_len]
                done = ep["dones"][s : s + seq_len]
            obs_b.append(obs)
            act_b.append(act)
            rew_b.append(rew)
            done_b.append(done)
        return np.array(obs_b), np.array(act_b), np.array(rew_b), np.array(done_b)

    def __len__(self):
        return len(self.episodes)


print("Data collection utilities defined.")
print("Testing data collection...")
test_eps = collect_pendulum_episodes(3, seed=42)
print(f"  Collected {len(test_eps)} test episodes")
print(f"  Obs shape: {test_eps[0]['obs'].shape}")
print(f"  Action shape: {test_eps[0]['actions'].shape}")
print(f"  Mean reward: {np.mean([ep['rewards'].sum() for ep in test_eps]):.1f}")

Data collection utilities defined.
Testing data collection...
  Collected 3 test episodes
  Obs shape: (200, 3)
  Action shape: (200, 1)
  Mean reward: -1254.5


---
## RSSM World Model Architecture

Standard RSSM architecture matching the project specification:  
- Encoder: obs_dim -> 512 -> 512 -> 512  
- Decoder: state_dim -> 512 -> 512 -> obs_dim  
- Reward predictor: state_dim -> 512 -> 512 -> 1  
- Continue predictor: state_dim -> 512 -> 512 -> 1  

This base model is used by all approaches. The `forward` method returns `(predictions, states_dict)` where
`states_dict` contains `'deter'`, `'stoch'`, `'priors'`, and `'posteriors'` -- matching the format
expected by `InterferenceEnsemble`.

In [3]:
"""
Cell: RSSM World Model Architecture
Purpose: Define the base world model used by all approaches
"""

class RSSMState(NamedTuple):
    deter: torch.Tensor
    stoch: torch.Tensor

    @property
    def combined(self):
        return torch.cat([self.deter, self.stoch], dim=-1)


class BaseWorldModel(nn.Module):
    """Standard RSSM world model matching the project architecture spec.

    Architecture:
        Encoder:  obs_dim -> 512 -> 512 -> 512
        Decoder:  state_dim -> 512 -> 512 -> obs_dim
        Reward:   state_dim -> 512 -> 512 -> 1
        Continue: state_dim -> 512 -> 512 -> 1
        GRU:      hidden_dim=512, deter_dim=512
        Prior:    deter_dim -> 512 -> stoch_dim*2
        Posterior: deter_dim+512 -> 512 -> stoch_dim*2

    forward() returns (predictions, states_dict) where states_dict has keys:
    'deter', 'stoch', 'priors', 'posteriors'
    """

    def __init__(
        self,
        obs_dim=OBS_DIM,
        action_dim=ACTION_DIM,
        stoch_dim=STOCH_DIM,
        deter_dim=DETER_DIM,
        hidden_dim=HIDDEN_DIM,
    ):
        super().__init__()
        self.obs_dim = obs_dim
        self.stoch_dim = stoch_dim
        self.deter_dim = deter_dim
        self.hidden_dim = hidden_dim
        self.state_dim = stoch_dim + deter_dim

        # Encoder: obs_dim -> 512 -> 512 -> 512
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Input projection: stoch + action -> hidden
        self.input_proj = nn.Sequential(
            nn.Linear(stoch_dim + action_dim, hidden_dim), nn.ELU()
        )

        # GRU dynamics
        self.gru = nn.GRUCell(hidden_dim, deter_dim)

        # Prior: deter -> stoch*2 (mean + log_std)
        self.prior_net = nn.Sequential(
            nn.Linear(deter_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, stoch_dim * 2),
        )

        # Posterior: deter + embed -> stoch*2
        self.posterior_net = nn.Sequential(
            nn.Linear(deter_dim + hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, stoch_dim * 2),
        )

        # Decoder: state -> obs
        self.decoder = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, obs_dim),
        )

        # Reward predictor: state -> 1
        self.reward_pred = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, 1),
        )

        # Continue predictor: state -> 1
        self.continue_pred = nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, 1),
        )

    def initial_state(self, batch_size, device):
        """Create initial RSSM state (zeros).

        Parameters
        ----------
        batch_size : int
            Batch size
        device : torch.device
            Device to create tensors on
        """
        return RSSMState(
            deter=torch.zeros(batch_size, self.deter_dim, device=device),
            stoch=torch.zeros(batch_size, self.stoch_dim, device=device),
        )

    def _get_dist(self, stats):
        mean, log_std = stats.chunk(2, dim=-1)
        std = F.softplus(log_std) + 0.1
        return torch.distributions.Normal(mean, std)

    def observe(self, obs, action, state):
        """Single-step observation update.

        Parameters
        ----------
        obs : torch.Tensor
            Observation at current step, shape (batch, obs_dim)
        action : torch.Tensor
            Action at current step, shape (batch, action_dim)
        state : RSSMState
            Previous RSSM state

        Returns
        -------
        Tuple of (new_state, prior_dist, posterior_dist)
        """
        embed = self.encoder(obs)
        x = self.input_proj(torch.cat([state.stoch, action], dim=-1))
        deter = self.gru(x, state.deter)
        posterior_stats = self.posterior_net(torch.cat([deter, embed], dim=-1))
        posterior = self._get_dist(posterior_stats)
        stoch = posterior.rsample()
        prior_stats = self.prior_net(deter)
        prior = self._get_dist(prior_stats)
        return RSSMState(deter, stoch), prior, posterior

    def decode(self, state):
        return self.decoder(state.combined)

    def forward(self, obs_seq, action_seq):
        """Forward pass through entire sequence.

        Parameters
        ----------
        obs_seq : torch.Tensor
            Observation sequence, shape (batch, seq_len, obs_dim)
        action_seq : torch.Tensor
            Action sequence, shape (batch, seq_len, action_dim)

        Returns
        -------
        Tuple of (predictions, states_dict)
            predictions: (batch, seq_len, obs_dim)
            states_dict: dict with 'deter', 'stoch', 'priors', 'posteriors'
        """
        batch_size, seq_len = obs_seq.shape[:2]
        device = obs_seq.device
        state = self.initial_state(batch_size, device)

        recon_obs, all_deter, all_stoch = [], [], []
        priors, posteriors = [], []

        for t in range(seq_len):
            state, prior, posterior = self.observe(
                obs_seq[:, t], action_seq[:, t], state
            )
            recon_obs.append(self.decode(state))
            all_deter.append(state.deter)
            all_stoch.append(state.stoch)
            priors.append(prior)
            posteriors.append(posterior)

        predictions = torch.stack(recon_obs, dim=1)
        states_dict = {
            "deter": torch.stack(all_deter, dim=1),
            "stoch": torch.stack(all_stoch, dim=1),
            "priors": priors,
            "posteriors": posteriors,
        }
        return predictions, states_dict


# Test model creation
test_model = BaseWorldModel().to(DEVICE)
num_params = sum(p.numel() for p in test_model.parameters())
print(f"BaseWorldModel created: {num_params:,} parameters")
del test_model

BaseWorldModel created: 4,732,677 parameters


---
## Quantum-Inspired Model Variants

- **EntanglementWorldModel**: BaseWorldModel with EntanglementLayer in the encoder
- **InterferenceWorldModel**: Wraps 5 BaseWorldModels via InterferenceEnsemble

In [4]:
"""
Cell: Quantum-Inspired Model Variants
Purpose: Define EntanglementWorldModel and InterferenceWorldModel
"""

class EntanglementWorldModel(BaseWorldModel):
    """RSSM with EntanglementLayer inserted into the encoder.

    The EntanglementLayer learns feature correlations between hidden
    dimensions, inspired by quantum entanglement. It is placed after
    the first hidden layer of the encoder.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        hidden_dim = kwargs.get("hidden_dim", HIDDEN_DIM)
        obs_dim = kwargs.get("obs_dim", OBS_DIM)
        # Replace encoder with entanglement-enhanced version
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ELU(),
            EntanglementLayer(dim=hidden_dim),
            nn.Linear(hidden_dim, hidden_dim), nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )


class InterferenceWorldModel(nn.Module):
    """Wrapper around InterferenceEnsemble that creates 5 BaseWorldModels
    and combines their predictions using quantum interference-inspired
    weighting.
    """

    def __init__(self, base_seed=None):
        super().__init__()
        self.ensemble = InterferenceEnsemble(
            model_class=BaseWorldModel,
            num_models=5,
            interference_strength=0.7,
            uncertainty_method="disagreement",
            base_seed=base_seed,
            obs_dim=OBS_DIM,
            action_dim=ACTION_DIM,
            stoch_dim=STOCH_DIM,
            deter_dim=DETER_DIM,
            hidden_dim=HIDDEN_DIM,
        )

    def forward(self, obs_seq, action_seq):
        return self.ensemble(obs_seq, action_seq, return_all=True)


def create_model(approach, seed=42):
    """Factory function to create the correct model for each approach.

    Parameters
    ----------
    approach : str
        One of: 'baseline', 'quantum_tunneling', 'superposition',
        'entanglement', 'interference_ensemble'
    seed : int
        Random seed for model initialization

    Returns
    -------
    nn.Module
        The model, moved to DEVICE
    """
    if approach == "entanglement":
        return EntanglementWorldModel(
            obs_dim=OBS_DIM, action_dim=ACTION_DIM,
            stoch_dim=STOCH_DIM, deter_dim=DETER_DIM, hidden_dim=HIDDEN_DIM,
        ).to(DEVICE)

    if approach == "interference_ensemble":
        return InterferenceWorldModel(base_seed=seed).to(DEVICE)

    # baseline, quantum_tunneling, superposition all use BaseWorldModel
    return BaseWorldModel(
        obs_dim=OBS_DIM, action_dim=ACTION_DIM,
        stoch_dim=STOCH_DIM, deter_dim=DETER_DIM, hidden_dim=HIDDEN_DIM,
    ).to(DEVICE)


# Verify all model variants
for approach in APPROACHES:
    m = create_model(approach)
    p = sum(p.numel() for p in m.parameters())
    print(f"  {approach}: {p:,} params")
    del m
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

  baseline: 4,732,677 params
  quantum_tunneling: 4,732,677 params
  superposition: 4,732,677 params
  entanglement: 5,258,629 params
  interference_ensemble: 23,663,391 params


---
## Loss Functions

Two loss functions:
- `compute_loss`: For single-model approaches (baseline, tunneling, superposition, entanglement)
- `compute_ensemble_loss`: For interference ensemble approach

In [5]:
"""
Cell: Loss Functions
Purpose: Define loss computation for standard and ensemble training
"""

def compute_loss(model, obs_seq, action_seq, reward_seq, kl_weight=KL_WEIGHT):
    """Standard RSSM loss for single-model approaches.

    Components:
        - Reconstruction loss (MSE between predicted and actual obs)
        - KL divergence between posterior and prior
        - Reward prediction loss (MSE)

    Parameters
    ----------
    model : BaseWorldModel
        The world model to compute loss for
    obs_seq : torch.Tensor
        Observation sequence, shape (batch, seq_len, obs_dim)
    action_seq : torch.Tensor
        Action sequence, shape (batch, seq_len, action_dim)
    reward_seq : torch.Tensor
        Reward sequence, shape (batch, seq_len)
    kl_weight : float
        Weight for KL divergence term

    Returns
    -------
    Dict[str, torch.Tensor]
        Dictionary with 'total', 'recon', 'kl', 'reward' loss tensors
    """
    predictions, states = model(obs_seq, action_seq)
    recon_loss = F.mse_loss(predictions, obs_seq)

    kl_losses = []
    for prior, posterior in zip(states["priors"], states["posteriors"]):
        kl = torch.distributions.kl_divergence(posterior, prior).sum(-1).mean()
        kl_losses.append(kl)
    kl_loss = (
        torch.stack(kl_losses).mean()
        if kl_losses
        else torch.tensor(0.0, device=obs_seq.device)
    )

    combined_states = torch.cat([states["deter"], states["stoch"]], dim=-1)
    reward_pred = model.reward_pred(combined_states)
    reward_loss = F.mse_loss(reward_pred.squeeze(-1), reward_seq)

    total = recon_loss + kl_weight * kl_loss + reward_loss
    return {
        "total": total,
        "recon": recon_loss,
        "kl": kl_loss,
        "reward": reward_loss,
    }


def compute_ensemble_loss(model, obs_seq, action_seq, reward_seq, kl_weight=KL_WEIGHT):
    """Loss for InterferenceEnsemble approach.

    Combines individual model losses, ensemble combined loss,
    KL divergence, and a diversity bonus that encourages model
    disagreement for better ensemble coverage.

    Parameters
    ----------
    model : InterferenceWorldModel
        The interference ensemble model
    obs_seq : torch.Tensor
        Observation sequence
    action_seq : torch.Tensor
        Action sequence
    reward_seq : torch.Tensor
        Reward sequence
    kl_weight : float
        Weight for KL divergence term

    Returns
    -------
    Dict[str, torch.Tensor]
        Dictionary with loss components
    """
    combined_pred, states = model(obs_seq, action_seq)
    all_predictions = states.get("all_predictions", None)

    combined_recon_loss = F.mse_loss(combined_pred, obs_seq)

    individual_loss = torch.tensor(0.0, device=obs_seq.device)
    if all_predictions is not None:
        individual_losses = [
            F.mse_loss(pred, obs_seq) for pred in all_predictions
        ]
        individual_loss = torch.stack(individual_losses).mean()

    kl_losses = []
    if "all_states" in states:
        for model_states in states["all_states"]:
            if "priors" in model_states and "posteriors" in model_states:
                for prior, posterior in zip(
                    model_states["priors"], model_states["posteriors"]
                ):
                    kl = (
                        torch.distributions.kl_divergence(posterior, prior)
                        .sum(-1)
                        .mean()
                    )
                    kl_losses.append(kl)
    kl_loss = (
        torch.stack(kl_losses).mean()
        if kl_losses
        else torch.tensor(0.0, device=obs_seq.device)
    )

    diversity = torch.tensor(0.0, device=obs_seq.device)
    if all_predictions is not None:
        mean_pred = all_predictions.mean(dim=0)
        diversity = ((all_predictions - mean_pred) ** 2).mean()

    # Negative diversity term encourages model diversity
    total = (
        0.5 * combined_recon_loss
        + 0.5 * individual_loss
        + kl_weight * kl_loss
        - 0.01 * diversity
    )
    return {
        "total": total,
        "combined_recon": combined_recon_loss,
        "individual_recon": individual_loss,
        "kl": kl_loss,
        "diversity": diversity,
    }


print("Loss functions defined.")

Loss functions defined.


---
## Training Functions

- `train_single_model`: Trains baseline, quantum_tunneling, superposition, and entanglement approaches
- `train_ensemble`: Trains the interference_ensemble approach

In [6]:
"""
Cell: Training Functions
Purpose: Training loops for single-model and ensemble approaches
"""
import pandas as pd


def train_single_model(model, buffer, approach, seed, num_steps=NUM_STEPS):
    """Train a single-model approach (baseline, tunneling, superposition, entanglement).

    Parameters
    ----------
    model : nn.Module
        The world model to train
    buffer : ReplayBuffer or SuperpositionReplayBuffer
        Experience replay buffer
    approach : str
        Name of the approach (affects optimizer choice and buffer sampling)
    seed : int
        Random seed (used for logging context)
    num_steps : int
        Number of gradient steps

    Returns
    -------
    pd.DataFrame
        Training history with columns: step, total, recon, kl, reward
    """
    # Select optimizer based on approach
    if approach == "quantum_tunneling":
        optimizer = QuantumTunnelingOptimizer(
            model.parameters(),
            lr=LEARNING_RATE,
            tunneling_strength=0.001,
            annealing_rate=0.9999,
            tunneling_frequency=100,
            min_tunneling=1e-8,
            stuck_threshold=500,
            base_optimizer="adamw",
        )
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    use_superposition_buffer = approach == "superposition"
    history = []

    for step in range(num_steps):
        model.train()

        # Sample batch from appropriate buffer type
        if use_superposition_buffer:
            batch = buffer.sample(batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
            obs = batch["obs"].to(DEVICE)
            actions = batch["actions"].to(DEVICE)
            rewards = batch["rewards"].to(DEVICE)
        else:
            obs_np, act_np, rew_np, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
            obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
            actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)
            rewards = torch.tensor(rew_np, dtype=torch.float32, device=DEVICE)

        optimizer.zero_grad()
        losses = compute_loss(model, obs, actions, rewards)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        if approach == "quantum_tunneling":
            optimizer.step(losses["total"].item())
        else:
            optimizer.step()

        history.append({
            "step": step,
            "total": losses["total"].item(),
            "recon": losses["recon"].item(),
            "kl": losses["kl"].item(),
            "reward": losses["reward"].item(),
        })

        if step % 1000 == 0:
            print(
                f"    Step {step}/{num_steps}: total={losses['total'].item():.4f} "
                f"recon={losses['recon'].item():.4f} kl={losses['kl'].item():.4f} "
                f"reward={losses['reward'].item():.4f}"
            )

    return pd.DataFrame(history)


def train_interference_ensemble(model, buffer, num_steps=NUM_STEPS):
    """Train interference ensemble approach.

    Parameters
    ----------
    model : InterferenceWorldModel
        The ensemble model to train
    buffer : ReplayBuffer
        Experience replay buffer
    num_steps : int
        Number of gradient steps

    Returns
    -------
    pd.DataFrame
        Training history
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history = []

    for step in range(num_steps):
        model.train()
        obs_np, act_np, rew_np, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
        obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
        actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)
        rewards = torch.tensor(rew_np, dtype=torch.float32, device=DEVICE)

        optimizer.zero_grad()
        losses = compute_ensemble_loss(model, obs, actions, rewards)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        history.append({
            "step": step,
            "total": losses["total"].item(),
            "combined_recon": losses["combined_recon"].item(),
            "individual_recon": losses["individual_recon"].item(),
            "kl": losses["kl"].item(),
            "diversity": losses["diversity"].item(),
        })

        if step % 1000 == 0:
            print(
                f"    Step {step}/{num_steps}: total={losses['total'].item():.4f} "
                f"combined={losses['combined_recon'].item():.4f} "
                f"kl={losses['kl'].item():.4f}"
            )

    return pd.DataFrame(history)


print("Training functions defined.")

Training functions defined.


---
## Evaluation Functions

Three evaluations per trained model:
1. **Train set evaluation**: Reconstruction MSE on training episodes
2. **Test set evaluation**: Reconstruction MSE on held-out episodes
3. **Long-horizon prediction**: Imagination accuracy at horizons [5, 10, 15, 20]

In [7]:
"""
Cell: Evaluation Functions
Purpose: Evaluate trained models on train set, test set, and long-horizon prediction
"""

def evaluate_model(model, buffer, approach, num_eval_batches=10):
    """Evaluate a model on data from the buffer.

    Computes observation reconstruction MSE and reward prediction MSE
    averaged over multiple batches.

    Parameters
    ----------
    model : nn.Module
        Trained world model
    buffer : ReplayBuffer
        Buffer containing evaluation data
    approach : str
        Approach name (affects how states are accessed)
    num_eval_batches : int
        Number of batches to average over

    Returns
    -------
    Dict[str, float]
        Dictionary with 'obs_mse' and 'reward_mse'
    """
    total_mse = 0.0
    total_reward_mse = 0.0
    count = 0

    with torch.no_grad():
        for _ in range(num_eval_batches):
            obs_np, act_np, rew_np, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
            obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
            actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)
            rewards = torch.tensor(rew_np, dtype=torch.float32, device=DEVICE)

            model.eval()
            pred, states = model(obs, actions)

            mse = F.mse_loss(pred, obs).item()
            total_mse += mse

            # Reward MSE
            if approach == "interference_ensemble":
                first_states = (
                    states["all_states"][0]
                    if "all_states" in states
                    else states
                )
                combined = torch.cat(
                    [first_states["deter"], first_states["stoch"]], dim=-1
                )
                rpred = (
                    model.ensemble.models[0]
                    .reward_pred(combined)
                    .squeeze(-1)
                )
            else:
                combined = torch.cat(
                    [states["deter"], states["stoch"]], dim=-1
                )
                rpred = model.reward_pred(combined).squeeze(-1)

            reward_mse = F.mse_loss(rpred, rewards).item()
            total_reward_mse += reward_mse
            count += 1

    return {
        "obs_mse": total_mse / count,
        "reward_mse": total_reward_mse / count,
    }


def evaluate_long_horizon(model, buffer, approach, horizons=None):
    """Evaluate prediction quality at different horizons.

    Tests how well the world model can predict future observations
    at increasing time horizons, which reveals whether the model
    has learned true environment dynamics or just short-term patterns.

    Parameters
    ----------
    model : nn.Module
        Trained world model
    buffer : ReplayBuffer
        Buffer containing evaluation data
    approach : str
        Approach name
    horizons : List[int], optional
        Horizons to evaluate at. Defaults to [5, 10, 15, 20].

    Returns
    -------
    Dict[int, float]
        MSE at each horizon
    """
    if horizons is None:
        horizons = [5, 10, 15, 20]
    max_horizon = max(horizons)

    results = {}
    with torch.no_grad():
        obs_np, act_np, _, _ = buffer.sample(BATCH_SIZE, max_horizon)
        obs = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE)
        actions = torch.tensor(act_np, dtype=torch.float32, device=DEVICE)

        if approach == "interference_ensemble":
            pred, _ = model(obs, actions)
        else:
            model.eval()
            pred, _ = model(obs, actions)

        for h in horizons:
            mse = F.mse_loss(pred[:, :h], obs[:, :h]).item()
            results[h] = mse

    return results


print("Evaluation functions defined.")

Evaluation functions defined.


---
## Run All Experiments

Running all 5 approaches x 5 seeds = 25 experiments.  
Each experiment: collect data -> train -> evaluate (train + test + long-horizon)

In [8]:
"""
Cell: Run All Experiments
Purpose: Execute all approach x seed combinations and save results
"""

all_results = []
summaries = {}

for approach in APPROACHES:
    print(f"\n{'='*70}")
    print(f"APPROACH: {approach}")
    print(f"{'='*70}")

    approach_results = []

    for seed in EXPERIMENT_SEEDS:
        print(f"\n  --- Seed {seed} ---")
        set_seed(seed)

        try:
            # Collect training data
            print(f"  Collecting {NUM_EPISODES} training episodes...")
            train_episodes = collect_pendulum_episodes(NUM_EPISODES, seed)
            print(f"  Collected {len(train_episodes)} episodes")

            # Create buffer
            if approach == "superposition":
                buffer = SuperpositionReplayBuffer(capacity=10000)
                for ep in train_episodes:
                    buffer.add(ep)
            else:
                buffer = ReplayBuffer(capacity=10000)
                for ep in train_episodes:
                    buffer.add(ep)

            # Create model
            model = create_model(approach, seed=seed)
            num_params = sum(p.numel() for p in model.parameters())
            print(f"  Model: {num_params:,} parameters")

            # Train
            start_time = time.time()
            if approach == "interference_ensemble":
                history = train_interference_ensemble(model, buffer)
            else:
                history = train_single_model(model, buffer, approach, seed)
            elapsed = time.time() - start_time
            print(f"  Training completed in {elapsed:.1f}s")

            # Evaluate on training data (first 50 episodes)
            eval_buffer = ReplayBuffer()
            for ep in train_episodes[:50]:
                eval_buffer.add(ep)
            train_metrics = evaluate_model(model, eval_buffer, approach)

            # Collect and evaluate on test data (different seed offset)
            print(f"  Collecting test episodes...")
            test_episodes = collect_pendulum_episodes(50, seed + 10000)
            test_buffer = ReplayBuffer()
            for ep in test_episodes:
                test_buffer.add(ep)
            test_metrics = evaluate_model(model, test_buffer, approach)

            # Long-horizon prediction evaluation
            horizon_results = evaluate_long_horizon(model, test_buffer, approach)

            result = {
                "approach": approach,
                "seed": seed,
                "train_obs_mse": train_metrics["obs_mse"],
                "train_reward_mse": train_metrics["reward_mse"],
                "test_obs_mse": test_metrics["obs_mse"],
                "test_reward_mse": test_metrics["reward_mse"],
                "long_horizon": {str(k): v for k, v in horizon_results.items()},
                "final_train_loss": float(history["total"].iloc[-1]),
                "time_seconds": elapsed,
                "num_params": num_params,
            }
            approach_results.append(result)
            all_results.append(result)

            # Save per-seed result
            seed_file = RESULTS_DIR / f"{approach}_seed_{seed}.json"
            with open(seed_file, "w") as f:
                json.dump(result, f, indent=2)
            print(f"  Saved: {seed_file.name}")
            print(f"  Test MSE: {result['test_obs_mse']:.6f}, Train MSE: {result['train_obs_mse']:.6f}")

            # Cleanup
            del model, buffer, eval_buffer, test_buffer
            torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

        except Exception as e:
            print(f"  ERROR: {e}")
            traceback.print_exc()
            approach_results.append({
                "approach": approach, "seed": seed, "error": str(e),
            })
            all_results.append(approach_results[-1])

    # Aggregate approach results
    valid = [r for r in approach_results if "error" not in r]
    if valid:
        test_mses = [r["test_obs_mse"] for r in valid]
        train_mses = [r["train_obs_mse"] for r in valid]
        reward_mses = [r["test_reward_mse"] for r in valid]
        times = [r["time_seconds"] for r in valid]

        # Aggregate long-horizon results
        long_horizon_means = {}
        for h_key in ["5", "10", "15", "20"]:
            h_vals = [
                r["long_horizon"][h_key]
                for r in valid
                if h_key in r.get("long_horizon", {})
            ]
            if h_vals:
                long_horizon_means[h_key] = {
                    "mean": float(np.mean(h_vals)),
                    "std": float(np.std(h_vals)),
                }

        summaries[approach] = {
            "test_obs_mse_mean": float(np.mean(test_mses)),
            "test_obs_mse_std": float(np.std(test_mses)),
            "train_obs_mse_mean": float(np.mean(train_mses)),
            "train_obs_mse_std": float(np.std(train_mses)),
            "test_reward_mse_mean": float(np.mean(reward_mses)),
            "test_reward_mse_std": float(np.std(reward_mses)),
            "time_mean": float(np.mean(times)),
            "time_std": float(np.std(times)),
            "num_params": valid[0]["num_params"],
            "num_seeds": len(valid),
            "long_horizon": long_horizon_means,
        }
        print(f"\n  {approach} Summary: Test MSE = {np.mean(test_mses):.6f} +/- {np.std(test_mses):.6f}")

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 70)


APPROACH: baseline

  --- Seed 42 ---
  Collected 100 episodes
  Model: 4,732,677 parameters
    Step 0/10000: total=56.7913 recon=4.4871 kl=0.6321 reward=51.6721
    Step 1000/10000: total=0.8486 recon=0.0704 kl=0.5983 reward=0.1798
    Step 2000/10000: total=0.6231 recon=0.0338 kl=0.4598 reward=0.1295
    Step 3000/10000: total=0.6421 recon=0.0438 kl=0.4414 reward=0.1568
    Step 4000/10000: total=0.6081 recon=0.0438 kl=0.4349 reward=0.1294
    Step 5000/10000: total=0.5381 recon=0.0267 kl=0.4112 reward=0.1002
    Step 6000/10000: total=0.5259 recon=0.0257 kl=0.4000 reward=0.1002
    Step 7000/10000: total=0.5311 recon=0.0319 kl=0.3852 reward=0.1140
    Step 8000/10000: total=0.5966 recon=0.0445 kl=0.4234 reward=0.1287
    Step 9000/10000: total=0.4833 recon=0.0197 kl=0.3962 reward=0.0674
  Training completed in 1181.7s
  Saved: baseline_seed_42.json
  Test MSE: 0.026358, Train MSE: 0.026358

  --- Seed 123 ---
  Collected 100 episodes
  Model: 4,732,677 parameters
    Step 0/10000:

---
## Save Complete Results

Save aggregated metrics as `complete_metrics.json` and print the final summary table.

In [9]:
"""
Cell: Save Complete Results
Purpose: Save aggregated results as complete_metrics.json
"""

complete_metrics = {
    "experiment": "phase1_pendulum_v1",
    "environment": {
        "name": "Pendulum-v1",
        "obs_dim": OBS_DIM,
        "action_dim": ACTION_DIM,
        "action_range": [-2.0, 2.0],
        "max_episode_length": 200,
    },
    "config": {
        "stoch_dim": STOCH_DIM,
        "deter_dim": DETER_DIM,
        "hidden_dim": HIDDEN_DIM,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "num_steps": NUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "kl_weight": KL_WEIGHT,
        "grad_clip": GRAD_CLIP,
        "num_episodes": NUM_EPISODES,
        "seeds": EXPERIMENT_SEEDS,
        "num_ensemble_models": 5,
        "interference_strength": 0.7,
    },
    "summary": summaries,
    "raw_results": all_results,
}

metrics_file = RESULTS_DIR / "complete_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(complete_metrics, f, indent=2, default=str)
print(f"Saved complete metrics: {metrics_file}")

# Print final summary table
print(f"\n{'='*70}")
print("PENDULUM-V1 RESULTS SUMMARY")
print(f"{'='*70}")
print(f"{'Approach':<25} {'Test MSE':>15} {'Train MSE':>15} {'Time (s)':>12} {'Params':>10}")
print("-" * 80)
for approach in APPROACHES:
    if approach in summaries:
        s = summaries[approach]
        print(f"{approach:<25} {s['test_obs_mse_mean']:.6f}+/-{s['test_obs_mse_std']:.4f} "
              f"{s['train_obs_mse_mean']:.6f}+/-{s['train_obs_mse_std']:.4f} "
              f"{s['time_mean']:>8.1f}+/-{s['time_std']:.1f} {s['num_params']:>10,}")
    else:
        print(f"{approach:<25} {'FAILED':>15}")

# Long-horizon summary
print(f"\n{'='*70}")
print("LONG-HORIZON PREDICTION MSE")
print(f"{'='*70}")
print(f"{'Approach':<25} {'h=5':>12} {'h=10':>12} {'h=15':>12} {'h=20':>12}")
print("-" * 75)
for approach in APPROACHES:
    if approach in summaries and summaries[approach].get("long_horizon"):
        lh = summaries[approach]["long_horizon"]
        vals = []
        for h_key in ["5", "10", "15", "20"]:
            if h_key in lh:
                vals.append(f"{lh[h_key]['mean']:.6f}")
            else:
                vals.append("   N/A")
        print(f"{approach:<25} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12} {vals[3]:>12}")

# Identify best approach
if summaries:
    best = min(summaries.items(), key=lambda x: x[1]["test_obs_mse_mean"])
    baseline_mse = summaries.get("baseline", {}).get("test_obs_mse_mean", float("nan"))
    if baseline_mse and not np.isnan(baseline_mse):
        improvement = (baseline_mse - best[1]["test_obs_mse_mean"]) / baseline_mse * 100
        print(f"\nBest approach: {best[0]} ({improvement:+.1f}% vs baseline)")

print(f"\nResults saved to: {RESULTS_DIR}")
print("Done.")

Saved complete metrics: d:\Git Repos\Quantum-Enhanced-Simulation-Learning-for-Reinforcement-Learning\experiments\results\phase1\pendulum\complete_metrics.json

PENDULUM-V1 RESULTS SUMMARY
Approach                         Test MSE       Train MSE     Time (s)     Params
--------------------------------------------------------------------------------
baseline                  0.027345+/-0.0026 0.026019+/-0.0027    952.6+/-247.8  4,732,677
quantum_tunneling         0.025967+/-0.0043 0.025776+/-0.0041    787.3+/-339.8  4,732,677
superposition             0.139595+/-0.0196 0.131192+/-0.0146    654.9+/-1.3  4,732,677
entanglement              0.031407+/-0.0077 0.029375+/-0.0065    651.5+/-1.5  5,258,629
interference_ensemble     0.030874+/-0.0021 0.030205+/-0.0030   2672.4+/-317.6 23,663,391

LONG-HORIZON PREDICTION MSE
Approach                           h=5         h=10         h=15         h=20
---------------------------------------------------------------------------
baseline            